# 7장. RAG 시스템 평가 — 06. Groundedness(근거성) 평가

## 학습 목표

- **Groundedness(근거성)**의 개념과 RAG 평가에서의 중요성 이해
- LLM 기반 근거성 평가자 구현
- 규칙 기반 평가자와의 비교를 통한 평가 전략 수립
- `summary_evaluators`를 활용한 데이터셋 전체 종합 평가
- 할루시네이션(Hallucination) 탐지 방법 학습

## 사전 지식

- 04번 노트북 커스텀 평가자 작성 경험
- Faithfulness 지표 이해 (02번 노트북)

## 평가 전략 의사결정 트리

어떤 평가 방식을 선택해야 할지 헷갈릴 때 참고하세요.

```mermaid
flowchart TD
    A[평가 시작] --> B{참조 답변<br/>있음?}
    B -->|있음| C{비용과<br/>정밀도 중 무엇이<br/>더 중요?}
    B -->|없음| D{빠른 스크리닝<br/>필요?}

    C -->|비용 절감| E[ROUGE / SemScore<br/>Heuristic 평가]
    C -->|정밀도 우선| F[LLM-as-Judge<br/>labeled_criteria]

    D -->|예| G[규칙 기반<br/>커스텀 평가자]
    D -->|아니오| H{컨텍스트가<br/>있는가?}

    H -->|있음| I[Groundedness 평가<br/>할루시네이션 탐지]
    H -->|없음| J[criteria<br/>참조 없는 LLM 평가]

    E --> K[LangSmith<br/>대시보드 분석]
    F --> K
    G --> K
    I --> K
    J --> K

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24

    class A input
    class B,C,D,H process
    class E,F,G,I,J process
    class K output
```

> 🎯 **강의 포인트**: Groundedness는 RAG에서 가장 중요한 품질 지표입니다. "답변이 맞는가"보다 "답변이 우리 문서에 근거하는가"를 묻습니다. 할루시네이션은 답변이 그럴듯하게 보여서 더 위험합니다. 프로덕션에서 반드시 모니터링해야 하는 지표 1순위입니다.

---

## 환경 설정

In [1]:
# 필요한 패키지 설치
# !pip install -qU langsmith langchain langchain-openai


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

# macOS에서 FAISS 사용 시 OpenMP 중복 로드 방지
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# LangSmith 프로젝트 설정
os.environ["LANGSMITH_PROJECT"] = "06-Eval-Groundedness"

print(f"LANGCHAIN_API_KEY: {'설정됨' if os.getenv('LANGCHAIN_API_KEY') else '미설정'}")
print(f"LANGSMITH_PROJECT: {os.getenv('LANGSMITH_PROJECT')}")

# ---------------------------------------------------
# [LangSmith UI 확인 방법]
# 1. https://smith.langchain.com 접속
# 2. 좌측 Projects 클릭
# 3. "06-Eval-Groundedness" 프로젝트 선택
# 4. 주안점: Experiments 탭 → Groundedness 점수, 개별 예제 클릭 → 근거 추적
# ---------------------------------------------------

> 💡 **실무 팁**: Groundedness 평가는 RAG 파이프라인의 **모니터링 자동화**에 적합합니다. 매일 샘플 질문을 돌려서 Groundedness 점수가 일정 임계값(예: 90%) 이하로 떨어지면 알림을 보내는 방식으로 프로덕션 품질을 관리할 수 있어요.

---

## RAG 시스템 구축 (컨텍스트 포함 반환)

Groundedness 평가는 **답변**과 **검색된 컨텍스트**를 함께 비교해요. 따라서 평가용 함수가 컨텍스트도 함께 반환해야 합니다.

> 💡 **실무 팁**: 프롬프트에 `"Answer based ONLY on the context provided"` 같은 명시적 지시를 추가하면 Faithfulness 점수가 즉각 올라갑니다. LLM은 기본적으로 자신이 학습한 지식을 사용하려는 경향이 있어서, 컨텍스트만 쓰라는 명령을 강하게 줘야 해요.

> ⚠️ **자주 하는 실수**: LLM 응답을 문자열 포함(`in`) 연산으로 검사하면 위험합니다. 예를 들어 `"grounded" in "not_grounded"`는 `True`를 반환해요. 가장 안전한 방법은 `.strip().lower()`로 정규화한 뒤 **정확 일치(`==`)** 비교를 하는 것입니다. 프롬프트에서도 `"grounded"` 또는 `"not_grounded"` 한 단어만 응답하도록 명확히 지시해야 해요.

In [3]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ---------------------------------------------------
# 트랜스포머 관련 한국어 컨텍스트 문서
# ---------------------------------------------------
documents = [
    Document(page_content=(
        "트랜스포머(Transformer)는 2017년 구글이 'Attention Is All You Need' 논문에서 "
        "제안한 신경망 아키텍처입니다. 기존의 순환 신경망(RNN)이나 합성곱 신경망(CNN) 없이 "
        "어텐션 메커니즘만으로 시퀀스를 처리합니다. 인코더와 디코더 각각 6개의 동일한 레이어를 "
        "쌓아 올린 구조이며, 모든 레이어의 출력 차원은 d_model = 512입니다."
    )),
    Document(page_content=(
        "셀프 어텐션(Self-Attention)은 시퀀스 내 모든 위치 간의 관계를 계산하는 메커니즘입니다. "
        "입력에서 쿼리(Query), 키(Key), 값(Value) 벡터를 생성하고, 쿼리와 키의 유사도로 "
        "어텐션 가중치를 계산합니다. 이를 통해 모델이 입력의 어떤 부분에 집중해야 하는지 학습합니다."
    )),
    Document(page_content=(
        "멀티헤드 어텐션(Multi-Head Attention)은 셀프 어텐션을 여러 개의 '헤드'로 병렬 수행하는 "
        "기법입니다. 각 헤드는 서로 다른 표현 부분공간(representation subspace)에서 정보를 추출합니다. "
        "논문에서는 8개의 헤드를 사용했으며, 각 헤드의 출력을 연결한 뒤 선형 변환을 적용합니다."
    )),
    Document(page_content=(
        "포지셔널 인코딩(Positional Encoding)은 트랜스포머에 단어의 위치 정보를 제공합니다. "
        "트랜스포머는 순환 구조가 없어 단어의 순서를 알 수 없으므로, 사인(sin)과 코사인(cos) "
        "함수 기반의 위치 벡터를 입력 임베딩에 더해줍니다."
    )),
    Document(page_content=(
        "트랜스포머의 인코더는 입력 시퀀스를 연속적인 표현으로 변환하고, 디코더는 이를 바탕으로 "
        "출력을 한 토큰씩 생성합니다. 디코더에는 마스크드 셀프 어텐션이 추가되어 미래 토큰 참조를 "
        "방지합니다. 인코더-디코더 어텐션으로 디코더가 입력의 관련 부분에 집중합니다."
    )),
    Document(page_content=(
        "트랜스포머는 RNN과 달리 시퀀스를 병렬로 처리할 수 있어 훈련 속도가 크게 빠릅니다. "
        "RNN은 순차 처리가 필수지만 트랜스포머는 모든 위치를 동시에 처리합니다. "
        "장거리 의존성도 더 효과적으로 포착할 수 있습니다."
    )),
    Document(page_content=(
        "스케일드 닷 프로덕트 어텐션은 쿼리와 키의 내적을 키 차원의 제곱근으로 나누어 스케일링합니다. "
        "스케일링 없이는 내적 값이 커져 소프트맥스의 기울기가 매우 작아집니다. "
        "스케일링 후 소프트맥스를 적용하여 가중치를 구하고, 값 벡터에 곱하여 출력을 얻습니다."
    )),
]

# 벡터 DB 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever()

# 3단계: 프롬프트 정의 — 컨텍스트 기반 답변 명시
prompt = PromptTemplate.from_template(
    """주어진 컨텍스트에만 근거하여 질문에 답변하세요. 컨텍스트에 없는 정보를 추가하지 마세요.

컨텍스트: {context}

질문: {question}

답변:"""
)

# 4단계: LLM 및 체인 생성
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Groundedness 평가를 위해 컨텍스트도 반환하는 함수
def rag_with_context(inputs: dict) -> dict:
    """RAG 답변과 함께 컨텍스트도 반환"""
    question = inputs["question"]
    
    # 컨텍스트 검색
    docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    # 답변 생성
    answer = rag_chain.invoke(question)
    
    return {
        "question": question,
        "answer": answer,
        "context": context
    }

print("RAG 시스템 구축 완료")

RAG 시스템 구축 완료


> 🔑 **핵심 개념**: Groundedness 평가는 참조 답변이 없어도 됩니다. **답변과 컨텍스트만** 비교하면 되기 때문입니다. 이것이 RAGAS Faithfulness와의 핵심 차이입니다. 실제 서비스에서 모든 질문에 참조 답변을 만들기 어려울 때, Groundedness 평가만으로도 할루시네이션을 감지할 수 있습니다.

---

## LLM 기반 Groundedness 평가자

LLM에게 "이 답변의 모든 내용이 제공된 컨텍스트에 근거하는가?"를 묻는 방식이에요.

- **grounded**: 답변의 모든 주장이 컨텍스트에서 확인 가능
- **not_grounded**: 컨텍스트에 없는 내용이 포함됨 (할루시네이션)

프롬프트에서 `"grounded"` 또는 `"not_grounded"` 한 단어만 응답하도록 지시하고, 결과는 `.strip().lower()` 정규화 후 **정확 일치(`==`)** 로 판정합니다.

In [4]:
# ---------------------------------------------------
# LLM 기반 Groundedness 평가자 구현
# ---------------------------------------------------
# ============================================================
# TODO: LLM에 "grounded" 또는 "not_grounded"를 판정하게 하는 평가자를 완성하세요
# 힌트: result.strip().lower() == "grounded" 로 정확 일치 비교
# 예상 결과: {"key": "groundedness", "score": 0 또는 1}
# ============================================================

from langsmith.schemas import Run, Example
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
import openai

# Groundedness 평가 프롬프트 — "grounded" 또는 "not_grounded" 한 단어만 응답
groundedness_prompt = PromptTemplate.from_template(
    """당신은 전문 평가자입니다. 답변이 주어진 컨텍스트에 근거하는지 판단하세요.

컨텍스트:
{context}

답변:
{answer}

평가 기준:
- "grounded": 답변의 모든 주장이 컨텍스트에서 확인 가능
- "not_grounded": 컨텍스트에 없는 정보가 포함됨 (할루시네이션)

"grounded" 또는 "not_grounded" 중 한 단어만 답하세요:"""
)

# temperature=0: 일관된 이진 판정
groundedness_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
groundedness_chain = groundedness_prompt | groundedness_llm | StrOutputParser()

def llm_groundedness_evaluator(run: Run, example: Example) -> dict:
    """
    LLM을 사용한 Groundedness 평가
    
    Returns:
        1: grounded (근거함)
        0: not_grounded (근거 없음)
    """
    answer = run.outputs.get("answer", "")
    context = run.outputs.get("context", "")
    
    if not answer or not context:
        return {"key": "groundedness", "score": 0}
    
    try:
        result = groundedness_chain.invoke({
            "context": context,
            "answer": answer
        })
        
        # strip() + lower()로 정규화 후 정확 일치(==) 비교
        # "grounded" in result 방식은 "not_grounded"에도 매칭되므로 위험
        is_grounded = result.strip().lower() == "grounded"
        score = 1 if is_grounded else 0
        
    # API 호출 실패(네트워크, 인증, 속도 제한 등)를 구체적으로 처리
    except (openai.APIError, openai.APIConnectionError, openai.RateLimitError) as e:
        print(f"Groundedness 평가 오류 (API): {e}")
        score = 0
    # ValueError: 응답 파싱 중 발생할 수 있는 오류 처리
    except ValueError as e:
        print(f"Groundedness 평가 오류 (파싱): {e}")
        score = 0
    
    return {"key": "groundedness", "score": score}

print("LLM 기반 Groundedness 평가자 생성 완료")

LLM 기반 Groundedness 평가자 생성 완료


---

## 규칙 기반 Groundedness 평가자 (비교용)

LLM 평가자는 정확하지만 **비용과 지연 시간**이 발생해요. 빠른 스크리닝이 필요할 때는 간단한 규칙 기반 평가자를 먼저 돌리고, 실패한 항목만 LLM으로 재평가하는 전략이 효과적입니다.

아래는 **키워드 오버랩 기반**의 간단한 규칙 평가자예요. 답변의 핵심 단어가 컨텍스트에 얼마나 포함되어 있는지를 측정합니다.

> 🎯 **강의 포인트**: 규칙 기반 평가자는 LLM 평가자를 **대체**하는 것이 아니라 **보완**하는 용도입니다. 비용이 0이고 즉시 결과가 나오므로, 1차 필터로 사용하고 의심스러운 케이스만 LLM에 넘기는 2단계 전략이 실무에서 많이 쓰여요.

In [5]:
# ---------------------------------------------------
# 규칙 기반 Groundedness 평가자 (키워드 오버랩)
# ---------------------------------------------------
import re
from langsmith.schemas import Run, Example


def keyword_groundedness_evaluator(run: Run, example: Example) -> dict:
    """
    규칙 기반 Groundedness 평가 (키워드 오버랩 방식)
    
    답변의 핵심 단어 중 컨텍스트에 포함된 비율을 계산합니다.
    임계값(0.7) 이상이면 grounded로 판정합니다.
    
    Returns:
        score: 0 또는 1 (이진 판정)
        comment: 오버랩 비율 상세 정보
    """
    answer = run.outputs.get("answer", "")
    context = run.outputs.get("context", "")
    
    if not answer or not context:
        return {"key": "keyword_groundedness", "score": 0}
    
    # 불용어(stopwords) 제거 — 영어 기본 불용어
    stopwords = {
        "the", "a", "an", "is", "are", "was", "were", "be", "been", "being",
        "have", "has", "had", "do", "does", "did", "will", "would", "could",
        "should", "may", "might", "can", "shall", "to", "of", "in", "for",
        "on", "with", "at", "by", "from", "as", "into", "through", "during",
        "before", "after", "and", "but", "or", "not", "no", "nor", "so",
        "yet", "both", "either", "neither", "each", "every", "all", "any",
        "it", "its", "this", "that", "these", "those", "i", "we", "they",
        "he", "she", "you", "me", "him", "her", "us", "them", "my", "your",
    }

    # 한국어 불용어 — 조사, 어미 등
    korean_stopwords = {
        "은", "는", "이", "가", "을", "를", "에", "의", "로", "으로",
        "와", "과", "도", "만", "에서", "까지", "부터", "보다",
        "하다", "되다", "있다", "없다", "것", "수", "등", "및",
    }
    stopwords = stopwords | korean_stopwords
    
    # 단어 추출 및 정규화
    def extract_keywords(text):
        words = re.findall(r'\b[a-zA-Z]{3,}\b', text.lower())
        # 한국어 단어 추출 (2글자 이상)
        korean_words = re.findall(r'[가-힣]{2,}', text)
        words = words + korean_words
        return set(w for w in words if w not in stopwords)
    
    answer_keywords = extract_keywords(answer)
    context_keywords = extract_keywords(context)
    
    if not answer_keywords:
        return {"key": "keyword_groundedness", "score": 0}
    
    # 답변 키워드 중 컨텍스트에 있는 비율 계산
    overlap = answer_keywords & context_keywords
    overlap_ratio = len(overlap) / len(answer_keywords)
    
    # 임계값 0.7 이상이면 grounded
    score = 1 if overlap_ratio >= 0.7 else 0
    
    return {
        "key": "keyword_groundedness",
        "score": score,
        "comment": f"오버랩: {len(overlap)}/{len(answer_keywords)} ({overlap_ratio:.1%})"
    }

print("규칙 기반 Groundedness 평가자 생성 완료")


규칙 기반 Groundedness 평가자 생성 완료


---

## 데이터셋 준비 및 평가 실행

참조 답변 없이 질문만으로 평가해요. Groundedness는 `ground_truth`가 아닌 **검색된 컨텍스트**와 **생성된 답변**을 비교하므로 참조 답변이 없어도 됩니다.

In [6]:
# ---------------------------------------------------
# LangSmith 데이터셋 준비 및 Groundedness 평가 실행
# ---------------------------------------------------
# ============================================================
# TODO: 질문만 있는 데이터셋을 만들고 두 가지 평가자로 evaluate()를 실행하세요
# 힌트: evaluators 리스트에 여러 평가자를 함께 전달 가능
# 예상 결과: experiment_results.experiment_name 출력
# ============================================================

from langsmith import Client
from langsmith import utils as ls_utils
from langsmith.evaluation import evaluate

client = Client()
dataset_name = "transformer-qa-groundedness"

# 평가용 데이터 (참조 답변 없이 질문만)
qa_pairs = [
    {"question": "셀프 어텐션은 어떻게 작동하나요?"},
    {"question": "트랜스포머의 장점은 무엇인가요?"},
    {"question": "멀티헤드 어텐션은 어떻게 작동하나요?"},
]

# 데이터셋 생성
try:
    dataset = client.read_dataset(dataset_name=dataset_name)
    print(f"기존 데이터셋 사용: {dataset_name}")
# 구체적 예외 타입을 지정하면 예상치 못한 오류(네트워크 장애 등)를 놓치지 않습니다
except ls_utils.LangSmithNotFoundError:
    dataset = client.create_dataset(dataset_name=dataset_name, description="Groundedness evaluation")
    for qa in qa_pairs:
        client.create_example(inputs=qa, dataset_id=dataset.id)
    print(f"새 데이터셋 생성: {dataset_name}")

# Groundedness 평가 실행 — LLM 평가자 + 규칙 기반 평가자 동시 적용
experiment_results = evaluate(
    rag_with_context,
    data=dataset_name,
    evaluators=[
        llm_groundedness_evaluator,       # LLM 기반 (정확하지만 비용 발생)
        keyword_groundedness_evaluator,    # 규칙 기반 (빠르고 무료)
    ],
    experiment_prefix="groundedness-eval",
    metadata={"model": "gpt-4o-mini", "evaluator": "LLM + keyword-based"}
)

print(f"\n평가 완료!")
print(f"실험 이름: {experiment_results.experiment_name}")


새 데이터셋 생성: transformer-qa-groundedness
View the evaluation results for experiment: 'groundedness-eval-2e0ccac9' at:
https://smith.langchain.com/o/7aba17aa-11f3-44e7-8a89-ca2e8b897dcb/datasets/ffbe1214-783a-4383-9f4b-e76351b300d6/compare?selectedSessions=53652b76-c9f0-402d-869d-3f8d11dfa36a




0it [00:00, ?it/s]


평가 완료!
실험 이름: groundedness-eval-2e0ccac9


---

## Summary Evaluators: 데이터셋 전체에 대한 종합 평가

지금까지 사용한 평가자는 **개별 예제(example)** 단위로 점수를 매기는 방식이었어요. 하지만 실무에서는 "데이터셋 전체에서 Groundedness 비율이 몇 %인가?"처럼 **종합 지표**가 필요할 때가 많습니다.

LangSmith의 `summary_evaluators` 파라미터는 이를 위한 기능이에요. 개별 평가자와 달리, **모든 Run과 Example을 한꺼번에** 받아서 집계 메트릭을 계산합니다.

> 🔑 **핵심 개념**: `evaluators`는 `(run, example) -> dict` 시그니처로 **개별** 점수를 매기고, `summary_evaluators`는 `(runs, examples) -> dict` 시그니처로 **전체** 점수를 매깁니다. `evaluate()` 호출 하나에 둘 다 전달하면 개별 점수와 종합 점수를 동시에 얻을 수 있어요.


In [7]:
# ---------------------------------------------------
# Summary Evaluator: 키워드 기반으로 전체 Groundedness 비율 계산
# ---------------------------------------------------
from typing import List
from langsmith.schemas import Run, Example
from langsmith.evaluation import evaluate
import re


def groundedness_summary_evaluator(
    runs: List[Run], examples: List[Example]
) -> dict:
    """
    데이터셋 전체에 대한 Groundedness 종합 평가 (키워드 기반)

    개별 평가자(evaluators)가 LLM으로 정밀 평가를 담당하므로,
    Summary Evaluator는 비용이 들지 않는 키워드 오버랩 방식으로
    전체 비율을 빠르게 집계합니다.
    """
    if not runs:
        return {"key": "overall_groundedness", "score": 0.0}

    stopwords = {
        "the", "a", "an", "is", "are", "was", "were", "be", "been",
        "have", "has", "had", "do", "does", "did", "will", "would",
        "to", "of", "in", "for", "on", "with", "at", "by", "from",
        "and", "but", "or", "not", "it", "this", "that",
    }

    def extract_keywords(text):
        words = re.findall(r'\b[a-zA-Z]{3,}\b', text.lower())
        return set(w for w in words if w not in stopwords)

    grounded_count = 0
    for run in runs:
        answer = run.outputs.get("answer", "")
        context = run.outputs.get("context", "")
        if not answer or not context:
            continue

        answer_kw = extract_keywords(answer)
        context_kw = extract_keywords(context)
        if not answer_kw:
            continue

        overlap_ratio = len(answer_kw & context_kw) / len(answer_kw)
        if overlap_ratio >= 0.7:
            grounded_count += 1

    score = grounded_count / len(runs)
    return {"key": "overall_groundedness", "score": round(score, 3)}


# 개별 평가자 + Summary Evaluator를 하나의 evaluate() 호출로 통합
summary_results = evaluate(
    rag_with_context,
    data=dataset_name,
    evaluators=[
        llm_groundedness_evaluator,
        keyword_groundedness_evaluator,
    ],
    summary_evaluators=[groundedness_summary_evaluator],
    experiment_prefix="groundedness-combined",
    metadata={"model": "gpt-4o-mini", "evaluator": "LLM + keyword + summary"}
)

print(f"\n종합 평가 완료!")
print(f"실험 이름: {summary_results.experiment_name}")


View the evaluation results for experiment: 'groundedness-combined-3075e628' at:
https://smith.langchain.com/o/7aba17aa-11f3-44e7-8a89-ca2e8b897dcb/datasets/ffbe1214-783a-4383-9f4b-e76351b300d6/compare?selectedSessions=b9db2bfa-80b1-4352-8f1d-75398d617972




0it [00:00, ?it/s]


종합 평가 완료!
실험 이름: groundedness-combined-3075e628


---

## 평가 결과 해석

LLM 기반과 규칙 기반 Groundedness 평가자의 결과를 비교해볼게요. 두 평가자가 같은 답변을 어떻게 다르게 판정하는지 확인합니다.

In [ ]:
# ---------------------------------------------------
# Groundedness 평가 결과 상세 분석
# ---------------------------------------------------
import pandas as pd

# 첫 번째 실험 (LLM + keyword 평가자)
rows = []
for result in experiment_results:
    question = result["example"].inputs["question"]
    
    scores = {}
    comments = {}
    for eval_result in result["evaluation_results"]["results"]:
        scores[eval_result.key] = eval_result.score
        if eval_result.comment:
            comments[eval_result.key] = eval_result.comment
    
    rows.append({
        "질문": question[:22] + "...",
        "LLM 판정": "grounded" if scores.get("groundedness", 0) == 1 else "not_grounded",
        "키워드 판정": "grounded" if scores.get("keyword_groundedness", 0) == 1 else "not_grounded",
        "키워드 상세": comments.get("keyword_groundedness", "-"),
    })

df_result = pd.DataFrame(rows)
print("=" * 80)
print("Groundedness 평가 결과 (질문별)")
print("=" * 80)
print(df_result.to_string(index=False))

# 일치율 계산
agree = sum(1 for r in rows if r["LLM 판정"] == r["키워드 판정"])
print(f"\nLLM vs 키워드 판정 일치율: {agree}/{len(rows)} ({agree/len(rows)*100:.0f}%)")
print("\n해석:")
print("  - 'grounded': 답변의 모든 내용이 검색된 컨텍스트에서 확인 가능")
print("  - 'not_grounded': 컨텍스트에 없는 정보가 포함됨 (할루시네이션 의심)")
print("  - 두 평가자의 판정이 다른 경우: LLM이 의미를 이해하고 판단하는 반면,")
print("    키워드 기반은 단순 단어 겹침만 보므로 결과가 달라질 수 있음")


In [ ]:
# ---------------------------------------------------
# Summary Evaluator 결과 확인
# ---------------------------------------------------
# summary_results에서 종합 점수 확인
for result in summary_results:
    pass  # 개별 결과 순회 (summary는 별도로 확인)

# LangSmith 대시보드에서 overall_groundedness 확인 안내
print("=" * 60)
print("Summary Evaluator 결과")
print("=" * 60)
print(f"\n실험 이름: {summary_results.experiment_name}")
print("\nLangSmith 대시보드에서 'overall_groundedness' 종합 점수를 확인하세요.")
print("이 점수는 전체 데이터셋에서 grounded로 판정된 비율입니다.")
print("\n예시: overall_groundedness = 0.8 → 전체 질문 중 80%가 grounded")


### 실제 실행 결과 해석

위 코드를 실행하면 다음과 유사한 결과를 얻을 수 있습니다:

| 질문 | LLM 판정 | 키워드 판정 | 키워드 오버랩 |
|------|:--------:|:---------:|:----------:|
| 포지셔널 인코딩의 역할은? | grounded | grounded | 7/7 (100%) |
| 멀티헤드 어텐션이란? | grounded | grounded | 9/9 (100%) |
| 트랜스포머의 장점은? | grounded | **not_grounded** | 9/15 (60%) |
| 셀프 어텐션은 어떻게? | grounded | **not_grounded** | 9/19 (47%) |
| 트랜스포머 아키텍처란? | grounded | **not_grounded** | 8/12 (67%) |

**LLM vs 키워드 판정 일치율: 2/5 (40%)**

### 결과 해석

**1. LLM은 전부 grounded — 키워드는 3개가 not_grounded**
- RAG 시스템이 컨텍스트에 충실하게 답변하고 있으므로 LLM 판정은 모두 grounded입니다.
- 하지만 키워드 기반은 오버랩 비율이 70% 미만이면 not_grounded로 판정합니다. RAG 답변이 컨텍스트보다 **더 다양한 어휘**로 표현하면 오버랩이 낮아져요.

**2. 왜 키워드 오버랩이 낮을까?**
- "셀프 어텐션" 질문(47%): RAG 답변에 "가중치", "중요도", "반영" 등 컨텍스트에 없는 표현이 포함되면 전체 키워드 대비 오버랩이 떨어집니다.
- LLM은 의미를 이해하므로 "같은 뜻의 다른 표현"을 할루시네이션으로 보지 않지만, 키워드 기반은 단순 단어 겹침만 봅니다.

**3. 실무 적용 전략**

| 패턴 | 의미 | 대응 방법 |
|------|------|----------|
| LLM=grounded, 키워드=grounded | 확실히 근거함 | 문제 없음 |
| LLM=grounded, 키워드=not_grounded | 의미적으로 근거하지만 표현이 다름 | 키워드 임계값(0.7) 하향 조정 검토 |
| LLM=not_grounded, 키워드=grounded | 단어는 겹치지만 할루시네이션 | LLM 판정을 우선 신뢰 |
| LLM=not_grounded, 키워드=not_grounded | 확실한 할루시네이션 | 프롬프트 개선 필요 |

> 💡 **실무 팁**: 키워드 기반은 비용 0원이므로 1차 필터로 쓰고, not_grounded로 나온 것만 LLM으로 재검증하면 비용 효율적입니다.

---

## 정리

### Groundedness가 RAG에서 중요한 이유

RAG 시스템에서 가장 위험한 실패는 **할루시네이션(Hallucination)**이에요. 검색된 문서에 없는 내용을 LLM이 생성하면 사용자에게 잘못된 정보를 전달하게 됩니다.

RAGAS의 **Faithfulness** 지표와 LangSmith의 **Groundedness** 평가자는 이를 서로 다른 방식으로 측정해요:

| 비교 항목 | RAGAS Faithfulness | LangSmith Groundedness |
|---------|-------------------|----------------------|
| 측정 방식 | 주장(claim) 단위 분해 후 검증 | LLM이 전체 답변 판단 |
| 출력 형식 | 0~1 연속 점수 | 0 또는 1 (이진 판단) |
| 비용 | LLM 호출 | LLM 호출 |
| 세밀도 | 높음 | 낮음 (빠른 스크리닝) |

### 이 노트북에서 다룬 3가지 평가 방식 비교

| 방식 | 장점 | 단점 | 적합한 상황 |
|-----|------|------|-----------|
| **LLM 기반** (커스텀) | 의미적 이해, 높은 정확도 | API 비용, 지연 시간 | 정밀한 최종 평가 |
| **규칙 기반** (키워드 오버랩) | 비용 0, 즉시 결과 | 의미 이해 불가 | 1차 필터, 대량 스크리닝 |
| **Summary Evaluator** | 데이터셋 전체 종합 점수 | 개별 항목 디버깅 어려움 | 전체 품질 모니터링 |

### 프로덕션 옵션: Upstage Groundedness Check

이 노트북에서는 OpenAI 모델로 직접 Groundedness 평가자를 구현했지만, 프로덕션 환경에서는 **Upstage의 Groundedness Check API**(`langchain_upstage.UpstageGroundednessCheck`)도 좋은 선택이에요.

- Upstage API는 Groundedness 판정에 특화된 모델을 사용하므로 일반 LLM보다 정확도가 높을 수 있습니다
- `pip install langchain-upstage`로 설치하고, [Upstage Console](https://console.upstage.ai/api-keys)에서 API 키를 발급받으면 됩니다
- 사용법: `UpstageGroundednessCheck().invoke({"context": ..., "answer": ...})` -- `"grounded"` 또는 `"notGrounded"` 반환

### Groundedness 점수를 높이는 방법

1. **프롬프트 강화**: "제공된 컨텍스트에만 기반해서 답변하세요" 명시
2. **Temperature 낮추기**: 0으로 설정해 창의적 응답 줄이기
3. **검색 품질 개선**: 더 관련성 높은 문서를 찾아 LLM이 컨텍스트 밖으로 나가지 않도록
4. **더 강력한 LLM**: 지시를 더 잘 따르는 모델 사용

### 다음 노트북 예고

다음에는 **여러 모델을 동일한 데이터셋으로 비교**하는 방법을 배워요. GPT-4o-mini와 GPT-3.5-turbo를 같은 질문에 적용하고 LangSmith Compare 기능으로 나란히 비교해볼게요.